# Phase 8: Feature Engineering

## Objective
- Load the target-labelled Mathematics and Portuguese datasets.
- Separate predictors from the target variable.
- Exclude G1, G2, and G3 to prevent grade-based target leakage.
- Inspect feature types and missing values.
- Save the feature datasets for the model development phase.

## Research safeguard
Feature transformations that learn parameters, such as scaling and imputation, must be fitted on training data only. Categorical encoding and scaling will be handled in the modelling pipeline after the train-test split.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Locate the project root whether Jupyter starts in the root
# folder or inside the notebooks folder.
current_dir = Path.cwd()

if (current_dir / "data" / "processed").exists():
    project_root = current_dir
elif (current_dir.parent / "data" / "processed").exists():
    project_root = current_dir.parent
else:
    raise FileNotFoundError(
        "Could not locate data/processed. "
        "Open the Student-Performance-Prediction project folder."
    )

processed_dir = project_root / "data" / "processed"
output_dir = processed_dir / "model_ready"
output_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Processed data:", processed_dir)
print("Output folder:", output_dir)
print("Output folder exists:", output_dir.exists())

Project root: e:\PROJECTS\Mini Project\Project\Student-Performance-Prediction
Processed data: e:\PROJECTS\Mini Project\Project\Student-Performance-Prediction\data\processed
Output folder: e:\PROJECTS\Mini Project\Project\Student-Performance-Prediction\data\processed\model_ready
Output folder exists: True


In [2]:
mat_path = processed_dir / "student_mat_target.csv"
por_path = processed_dir / "student_por_target.csv"

for path in [mat_path, por_path]:
    if not path.exists():
        raise FileNotFoundError(f"Required dataset not found: {path}")

mat_df = pd.read_csv(mat_path)
por_df = pd.read_csv(por_path)

print("Mathematics dataset shape:", mat_df.shape)
print("Portuguese dataset shape:", por_df.shape)

print("\nMathematics columns:")
print(mat_df.columns.tolist())

print("\nPortuguese columns:")
print(por_df.columns.tolist())

print("\nMathematics preview:")
display(mat_df.head())

print("\nPortuguese preview:")
display(por_df.head())

Mathematics dataset shape: (395, 34)
Portuguese dataset shape: (649, 34)

Mathematics columns:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3', 'performance']

Portuguese columns:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3', 'performance']

Mathematics preview:


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3,performance
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,3,4,1,1,3,6,5,6,6,0
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,3,3,1,1,3,4,5,5,6,0
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,3,2,2,3,3,10,7,8,10,1
3,GP,F,15,U,GT3,T,4,2,health,services,...,2,2,1,1,5,2,15,14,15,1
4,GP,F,16,U,GT3,T,3,3,other,other,...,3,2,1,2,5,4,6,10,10,1



Portuguese preview:


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3,performance
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,3,4,1,1,3,4,0,11,11,1
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,3,3,1,1,3,2,9,11,11,1
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,3,2,2,3,3,6,12,13,12,1
3,GP,F,15,U,GT3,T,4,2,health,services,...,2,2,1,1,5,0,14,14,14,1
4,GP,F,16,U,GT3,T,3,3,other,other,...,3,2,1,2,5,0,11,13,13,1


In [3]:
target_col = "performance"

for name, df in [
    ("Mathematics", mat_df),
    ("Portuguese", por_df)
]:
    print(f"\n{name} dataset")

    assert target_col in df.columns, (
        f"Missing '{target_col}' column in {name} dataset"
    )

    print("Missing target values:", df[target_col].isna().sum())
    print("Target values:", sorted(df[target_col].dropna().unique()))
    print("Target distribution:")
    display(df[target_col].value_counts(dropna=False).sort_index())

    assert df[target_col].notna().all(), (
        f"{name} dataset has missing target values"
    )


Mathematics dataset
Missing target values: 0
Target values: [np.int64(0), np.int64(1)]
Target distribution:


performance
0    130
1    265
Name: count, dtype: int64


Portuguese dataset
Missing target values: 0
Target values: [np.int64(0), np.int64(1)]
Target distribution:


performance
0    100
1    549
Name: count, dtype: int64

In [4]:
target_col = "performance"
grade_cols = ["G1", "G2", "G3"]

def prepare_features(df, dataset_name):
    df = df.copy()

    # The outcome must exist before selecting predictors.
    if target_col not in df.columns:
        raise ValueError(
            f"{dataset_name}: '{target_col}' is missing."
        )

    # Exclude grades that could reveal the outcome.
    excluded_grades = [
        col for col in grade_cols if col in df.columns
    ]

    X = df.drop(columns=[target_col] + excluded_grades)
    y = df[target_col].astype(int)

    # Confirm that the predictors do not contain the target
    # or the excluded grade columns.
    assert target_col not in X.columns
    assert not any(col in X.columns for col in grade_cols)

    print(f"\n{dataset_name}")
    print("Rows:", len(X))
    print("Number of features:", X.shape[1])
    print("Excluded grade columns:", excluded_grades)
    print("Target distribution:")
    print(y.value_counts().sort_index())

    return X, y

X_mat, y_mat = prepare_features(mat_df, "Mathematics")
X_por, y_por = prepare_features(por_df, "Portuguese")

print("\nMathematics feature names:")
print(X_mat.columns.tolist())

print("\nPortuguese feature names:")
print(X_por.columns.tolist())


Mathematics
Rows: 395
Number of features: 30
Excluded grade columns: ['G1', 'G2', 'G3']
Target distribution:
performance
0    130
1    265
Name: count, dtype: int64

Portuguese
Rows: 649
Number of features: 30
Excluded grade columns: ['G1', 'G2', 'G3']
Target distribution:
performance
0    100
1    549
Name: count, dtype: int64

Mathematics feature names:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences']

Portuguese feature names:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absen

In [5]:
for name, X in [
    ("Mathematics", X_mat),
    ("Portuguese", X_por)
]:
    print(f"\n{'=' * 15} {name} {'=' * 15}")

    print("\nData types:")
    display(X.dtypes.astype(str).value_counts())

    print("\nMissing values per column:")
    missing = X.isna().sum()
    display(missing[missing > 0].sort_values(ascending=False))

    print("\nDuplicate predictor rows:", X.duplicated().sum())

    print("\nNumeric feature summary:")
    display(X.describe(include=[np.number]).T)

    print("\nCategorical feature summary:")
    display(X.describe(include=["object", "category"]).T)


=============== Mathematics ===============

Data types:


object    17
int64     13
Name: count, dtype: int64


Missing values per column:


Series([], dtype: int64)


Duplicate predictor rows: 0

Numeric feature summary:


...

Series([], dtype: int64)


Duplicate predictor rows: 0

Numeric feature summary:


,count,mean,std,min,25%,50%,75%,max
age,649.0,16.744222,1.218138,15.0,16.0,17.0,18.0,22.0
Medu,649.0,2.514638,1.134552,0.0,2.0,2.0,4.0,4.0
Fedu,649.0,2.306626,1.099931,0.0,1.0,2.0,3.0,4.0
traveltime,649.0,1.568567,0.748660,1.0,1.0,1.0,2.0,4.0
studytime,649.0,1.930663,0.829510,1.0,1.0,2.0,2.0,4.0
failures,649.0,0.221880,0.593235,0.0,0.0,0.0,0.0,3.0
famrel,649.0,3.930663,0.955717,1.0,4.0,4.0,5.0,5.0
freetime,649.0,3.180277,1.051093,1.0,3.0,3.0,4.0,5.0
goout,649.0,3.184900,1.175766,1.0,2.0,3.0,4.0,5.0
Dalc,649.0,1.502311,0.924834,1.0,1.0,1.0,2.0,5.0



Categorical feature summary:


,count,unique,top,freq
school,649,2,GP,423
sex,649,2,F,383
address,649,2,U,452
famsize,649,2,GT3,457
Pstatus,649,2,T,569
Mjob,649,5,other,258
Fjob,649,5,other,367
reason,649,4,course,285
guardian,649,3,mother,455
schoolsup,649,2,no,581


In [6]:
def save_feature_dataset(X, y, output_path):
    result = X.copy()
    result[target_col] = y.to_numpy()

    result.to_csv(output_path, index=False)

    # Reload to verify that the file was saved correctly.
    saved = pd.read_csv(output_path)

    assert len(saved) == len(result)
    assert target_col in saved.columns
    assert saved[target_col].notna().all()

    print(f"\nSaved: {output_path}")
    print("Shape:", saved.shape)
    print("Target counts:")
    print(saved[target_col].value_counts().sort_index())

    return saved

mat_features_path = output_dir / "student_mat_features.csv"
por_features_path = output_dir / "student_por_features.csv"

mat_features = save_feature_dataset(
    X_mat, y_mat, mat_features_path
)

por_features = save_feature_dataset(
    X_por, y_por, por_features_path
)


Saved: e:\PROJECTS\Mini Project\Project\Student-Performance-Prediction\data\processed\model_ready\student_mat_features.csv
Shape: (395, 31)
Target counts:
performance
0    130
1    265
Name: count, dtype: int64

Saved: e:\PROJECTS\Mini Project\Project\Student-Performance-Prediction\data\processed\model_ready\student_por_features.csv
Shape: (649, 31)
Target counts:
performance
0    100
1    549
Name: count, dtype: int64


In [7]:
for name, df in [
    ("Mathematics", mat_features),
    ("Portuguese", por_features)
]:
    forbidden = [
        col for col in grade_cols if col in df.columns
    ]

    assert target_col in df.columns
    assert not forbidden, (
        f"{name} still contains grade columns: {forbidden}"
    )
    assert df[target_col].isin([0, 1]).all()

    print(f"\n{name}: verification passed")
    print("Rows:", len(df))
    print("Features:", df.shape[1] - 1)
    print("Target distribution:")
    print(df[target_col].value_counts().sort_index())

print("\nFeature engineering completed successfully.")


Mathematics: verification passed
Rows: 395
Features: 30
Target distribution:
performance
0    130
1    265
Name: count, dtype: int64

Portuguese: verification passed
Rows: 649
Features: 30
Target distribution:
performance
0    100
1    549
Name: count, dtype: int64

Feature engineering completed successfully.
